In [ ]:
#investigative cell.
import pandas as pd
import numpy as np

df = pd.read_csv('Data/accidental_drugs.csv')
print(df.head())
print(df.info())

print(df['DateType'].value_counts())
print('\n---')
print(df.groupby('DateType')['Age'].count())

print(df['Heroin'].value_counts(dropna=False))
print('\n---')
print(df['Fentanyl'].value_counts(dropna=False))
print('\n---')
print(df['AnyOpioid'].value_counts(dropna=False))

#what's the structure of the geographic format?
print(df[['DeathCityGeo', 'ResidenceCityGeo']])


In [ ]:
#cleaning cell.
#1. create a dictionary for reference.
data_dictionary = {
    'ID': 'Case identifier (format: YY-NNNN)',
    'Date': 'Date value - check DateType to know if this is death or report date',
    'DateType': 'Indicates whether Date is "DateofDeath" or "DateReported"',
    'Age': 'Age at death (numeric, years)',
    'Sex': 'Gender (Male/Female)',
    'Race': 'Race category',
    'ResidenceCity/County/State': 'Location where victim lived',
    'DeathCity/County': 'Location where death occurred',
    'InjuryCity/County/State': 'Location where injury occurred (may differ from death location)',
    'COD': 'Cause of death (text)',
    'Drug columns': 'Indicates drug found in body. "Y" = present, NaN = not detected or not tested',
    'AnyOpioid': 'Aggregate indicator for any opioid presence ("Y"/"N"/NaN)',
    'MannerofDeath': 'All "Accident" in this dataset',
    '*Geo columns': 'Coordinates embedded in text: "City, ST\n(lat, lon)"'
}

for col_name, description in data_dictionary.items():
    print(f'{col_name}: {description}')


#2. Date processing

df['Date'] = pd.to_datetime(df['Date'])
df['year'] = df['Date'].dt.year
df['month'] = df['Date'].dt.month
df['quarter'] = df['Date'].dt.quarter
#NB: 45% of our records use report date, not the death date.

print(df.info())

#keep and original copy for later
df_drug_original = df.copy()

# True drug columns only (based on your data)
drug_columns = [
    'Heroin', 'Cocaine', 'Fentanyl', 'FentanylAnalogue',
    'Oxycodone', 'Oxymorphone', 'Ethanol', 'Hydrocodone',
    'Benzodiazepine', 'Methadone', 'Amphet', 'Tramad',
    'Morphine_NotHeroin', 'Hydromorphone', 'OpiateNOS'
]

# Verify they exist
existing_drugs = [col for col in drug_columns if col in df.columns]
print(f"Found {len(existing_drugs)} drug columns to process")

def clean_and_convert_drugs(df, drug_columns):
    """
    Step 1: Convert any value starting with 'Y' to 'Y'
    Step 2: Convert 'Y' → 1, everything else → 0
    """
    df_clean = df.copy()
    
    for col in drug_columns:
        # Step 1: Anything starting with 'Y' becomes 'Y' (cleans Y-A, Y POPS, etc.)
        df_clean[col] = df_clean[col].str.startswith('Y', na=False)
        
        # Step 2: Convert True/False to 1/0
        df_clean[col] = df_clean[col].astype(int)
    
    return df_clean

# Apply the function
df = clean_and_convert_drugs(df, existing_drugs)

# Verify
print("\nFirst 5 rows of cleaned drug columns:")
print(df[existing_drugs].head())
print("\nValue counts for Heroin (should be only 0 and 1):")
print(df['Heroin'].value_counts())






In [ ]:
#cleaning cont.
#convert AnyOpioid separately
df['AnyOpioid'] = df['AnyOpioid'].str.startswith('Y', na=False).astype(int)

print("\nAnyOpioid prevalence:")
print(f"  Any opioid detected: {df['AnyOpioid'].mean()*100:.1f}%")

In [ ]:
#what proportion of our victims died in their residence cities, and what proportion migrated?
#1. compare ResidenceCity vs DeathCity for all rows.
#standardise city names for comparison.

def standardise_city(city):
    '''Convert city to capital letters and remove space'''
    if pd.isna(city):
        return None
    return str(city).strip().upper()

#apply standardisation.
df['ResidenceCity_clean'] = df['ResidenceCity'].apply(standardise_city)
df['DeathCity_clean'] = df['DeathCity'].apply(standardise_city)

#create a flag; Did they die in their residence city?
df['died_in_residence_city'] = (df['ResidenceCity_clean'] == df['DeathCity_clean'])

#count rows where both cities are known.
both_known = df['ResidenceCity_clean'].notna() & df['DeathCity_clean'].notna()
print(f"Rows with both residence and death city known: {both_known.sum()}/{len(df)} ({both_known.sum()/len(df)*100:.1f}%)")

#among those, how many dies at home?
died_at_home = df[both_known]['died_in_residence_city'].sum()
print(f'Died in residence city: {died_at_home}/{both_known.sum()} ({died_at_home/both_known.sum()*100:.1f}%)')

#how many died 'FAR FROM HOME'
died_elsewhere = both_known.sum() - died_at_home
print(f"Died elsewhere: {died_elsewhere}/{both_known.sum()} ({died_elsewhere/both_known.sum()*100:.1f}%)")

#calculate the proportion who migrated (died elsewhere)
migration_rate = died_elsewhere / both_known.sum() * 100
print(f"\nMIGRATION RATE: {migration_rate:.1f}% of victims died outside their residence city")

if 25 < migration_rate < 40:
    print("✓ Matches the 'roughly one third' pattern I read about")
elif migration_rate > 40:
    print("Higher than expected - more migration than typical")
else:
    print("Lower than expected - most died at home")

#show examples of people who died in a different city
mismatch_df = df[both_known & ~df['died_in_residence_city']][['ResidenceCity', 'DeathCity']].drop_duplicates()

print("\nExample of city mismatches (first 10):")
print(mismatch_df.head(10))

#most common death cities for non-residents
print("\nTop 10 death cities for non-residents:")
non_resident_deaths = df[both_known & ~df['died_in_residence_city']]
print(non_resident_deaths['DeathCity'].value_counts().head(10))

#Create a clean migration flag (1 = died outside residence city, 0 = died at home or unknown)
df['migrated'] = 0
df.loc[both_known & ~df['died_in_residence_city'], 'migrated'] = 1

#Check distribution
print("\nMigration flag distribution (1 = died elsewhere, 0 = died at home or unknown):")
print(df['migrated'].value_counts())

#Some more...
#rug prevalence per city for top 20 cities(by death count)
top_cities = df['DeathCity'].value_counts().head(20).index.tolist()

#create a prevalence table
prevalence_by_city = []

for city in top_cities:
    city_data = df[df['DeathCity'] == city]
    city_prevalence = {
        'City': city,
        'Total_Deaths': len(city_data)
    }

    #add prevalence for each drug
    for drug in existing_drugs:
        city_prevalence[f'{drug}_pct'] = city_data[drug].mean() * 100

    prevalence_by_city.append(city_prevalence)

#convert to DataFrame
prevalence_df = pd.DataFrame(prevalence_by_city)

#top 10 drugs by average prevalence across cities
print("=== DRUG PREVALENCE PER CITY ===\n")
print("Top 10 cities by death count:")
print(prevalence_df[['City', 'Total_Deaths'] + [f'{drug}_pct' for drug in existing_drugs[:5]]].head(10))

# Find which drug dominates each city
print("\n=== MOST COMMON DRUG BY CITY ===\n")
for city in top_cities[:10]:
    city_row = prevalence_df[prevalence_df['City'] == city]
    drug_cols = [f'{drug}_pct' for drug in existing_drugs]
    max_drug = city_row[drug_cols].max(axis=1).values[0]
    max_drug_name = city_row[drug_cols].idxmax(axis=1).values[0].replace('_pct', '')
    print(f"{city}: {max_drug_name} ({max_drug:.1f}%)")


#drug profiles: migrants vs. locals.
print("\n=== DRUG PROFILES: MIGRANTS VS NON-MIGRANTS ===\n")

# Split data
migrants = df[df['migrated'] == 1]
non_migrants = df[df['migrated'] == 0]

print(f"Migrants: {len(migrants)} cases")
print(f"Non-migrants: {len(non_migrants)} cases\n")

# Compare drug prevalence
comparison = []
for drug in existing_drugs:
    migrant_pct = migrants[drug].mean() * 100
    non_migrant_pct = non_migrants[drug].mean() * 100
    diff = migrant_pct - non_migrant_pct
    comparison.append({
        'Drug': drug,
        'Migrants (%)': round(migrant_pct, 1),
        'Non-Migrants (%)': round(non_migrant_pct, 1),
        'Difference': round(diff, 1)
    })

comparison_df = pd.DataFrame(comparison)
comparison_df = comparison_df.sort_values('Difference', ascending=False)

print("Drugs MORE common among migrants (positive = migrants use more):")
print(comparison_df.head(10))

print("\nDrugs LESS common among migrants (negative = migrants use less):")
print(comparison_df.tail(10))

# Key insight
print(f"\n🔍 KEY INSIGHT: Migrants are {comparison_df.iloc[0]['Difference']:.1f}% more likely to have {comparison_df.iloc[0]['Drug']}")



In [ ]:
#map migration patterns
import matplotlib.pyplot as plt
import seaborn as sns

#setting up visualisation style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("RdYlBu_r")

# Figure 1: Migration rate by city
print("\n GENERATING VISUALISATIONS \n")

#calculating migration rate per death city
city_migration = []
for city in top_cities:
    city_data = df[df['DeathCity'] == city]
    city_known = city_data[city_data['ResidenceCity_clean'].notna()]
    if len(city_known) > 10:  # Only cities with enough data
        migrant_rate = city_known['migrated'].mean() * 100
        city_migration.append({
            'City': city,
            'Total_Deaths': len(city_data),
            'Migration_Rate': migrant_rate
        })

migration_df = pd.DataFrame(city_migration).sort_values('Migration_Rate', ascending=False)

# Plot 1: Top 10 cities by migration rate
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: Highest migration rates
top_migration = migration_df.head(10)
axes[0].barh(top_migration['City'], top_migration['Migration_Rate'], color='coral', edgecolor='black')
axes[0].set_xlabel('Migration Rate (%)')
axes[0].set_title('Cities Where Non-Residents Most Frequently Die')
axes[0].invert_yaxis()
for i, v in enumerate(top_migration['Migration_Rate']):
    axes[0].text(v + 1, i, f'{v:.1f}%', va='center')

# Right: Lowest migration rates (people die at home)
low_migration = migration_df.tail(10)
axes[1].barh(low_migration['City'], low_migration['Migration_Rate'], color='steelblue', edgecolor='black')
axes[1].set_xlabel('Migration Rate (%)')
axes[1].set_title('Cities Where Residents Most Often Die at Home')
axes[1].invert_yaxis()
for i, v in enumerate(low_migration['Migration_Rate']):
    axes[1].text(v + 1, i, f'{v:.1f}%', va='center')

plt.tight_layout()
plt.show()

# Plot 2: Geographic heatmap (simplified - top cities by migrant count)
fig, ax = plt.subplots(figsize=(12, 6))

#getting top 15 cities by migrant count
migrant_counts = df[df['migrated'] == 1]['DeathCity'].value_counts().head(15)
cities = migrant_counts.index
counts = migrant_counts.values

bars = ax.bar(cities, counts, color='darkred', edgecolor='black', alpha=0.7)
ax.set_xlabel('City')
ax.set_ylabel('Number of Migrant Deaths')
ax.set_title('Cities Receiving Most Non-Resident Deaths')
ax.tick_params(axis='x', rotation=45)

# Add value labels
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, 
            str(count), ha='center', va='bottom')

plt.tight_layout()
plt.show()


In [ ]:
#cluster analysis
#check available data for clustering
print(' Clustering Data Check \n')

#featyres to use
demographic_cols = ['Age', 'Sex', 'Race']
drug_cols = existing_drugs
migration_col = 'migrated'

#check completeness
for col in demographic_cols + drug_cols + [migration_col]:
    if col in df.columns:
        non_null = df[col].notna().sum()
        print(f"{col}: {non_null}/{len(df)} ({non_null/len(df)*100:.1f}%)")
    else:
        print(f"{col}: NOT FOUND")

# Check how many rows have ALL features (complete cases)
features_for_clustering = demographic_cols + drug_cols
complete_cases = df[features_for_clustering].dropna().shape[0]
print(f"\nComplete cases for demographics vs. drugs: {complete_cases}/{len(df)} ({complete_cases/len(df)*100:.1f}%)")

#let's cluster
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA

#data preparation
print('1. Data preparation.')
#select features
demographic_cols = ['Age', 'Sex', 'Race']
drug_cols = existing_drugs
migration_col = 'migrated'

#encode categorical variables (sex, race)
df_encoded = df.copy()

#encode sex: Male = 1, Female = 0
df_encoded['Sex_encoded'] = (df_encoded['Sex'] == 'Male').astype(int)

#encode race: encode top races, group others as other.
top_races = df_encoded['Race'].value_counts().head(4).index.tolist()
for race in top_races:
    df_encoded[f'Race_{race}'] = (df_encoded['Race'] == race).astype(int)
# Add 'Other' race category
df_encoded['Race_Other'] = (~df_encoded['Race'].isin(top_races) & df_encoded['Race'].notna()).astype(int)

#define feature sets
features_C = ['Age', 'Sex_encoded'] + [f'Race_{r}' for r in top_races] + ['Race_Other'] + drug_cols
features_D = features_C + [migration_col]

#drop rows with missing values
df_cluster = df_encoded[features_D].dropna()
print(f"Rows for clustering: {len(df_cluster)}")

#scaling
print('2. Scaling.')
#check scale of features.
print('Feature ranges before scaling')
for col in ['Age'] + drug_cols[:2]:
    print(f' {col}: min={df_cluster[col].min():.2f}, max={df_cluster[col].max():.2f}')

#scaling because age(0-100) dominates drug columns (0-1)
scaler = StandardScaler()
X_C_scaled = scaler.fit_transform(df_cluster[features_C])
X_D_scaled = scaler.fit_transform(df_cluster[features_D])

print("Scaling applied (Age would otherwise dominate clustering)")

#find optimal k (elbow and silhouette)
print('\n - 3. Finding optimal K - \n ')
k_range = range(2, 11)
inertias = []
silhouette_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_C_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_C_scaled, kmeans.labels_))

#plot elbow and silhouette
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(x=4, color='r', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')

axes[1].plot(k_range, silhouette_scores, 'ro-', linewidth=2, markersize=8)
axes[1].axvline(x=4, color='r', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score (higher = better)')

plt.tight_layout()
plt.show()

#recommedn k
best_k_silhouette = k_range[np.argmax(silhouette_scores)]
print(f"\n Silhouette score suggests k={best_k_silhouette}")

#run clusters for Demographics + Drugs, k=4
print('\n - 4. Drugs+Demograaphics clustering \n')

k = 4
kmeans_C = KMeans(n_clusters=k, random_state=42, n_init=10)
clusters_C = kmeans_C.fit_predict(X_C_scaled)
df_cluster['Cluster_C'] = clusters_C

# Cluster profiles
print(f"Cluster sizes (k={k}):")
print(df_cluster['Cluster_C'].value_counts().sort_index())

# Calculate cluster characteristics
cluster_profiles_C = []
for i in range(k):
    cluster_data = df_cluster[df_cluster['Cluster_C'] == i]
    profile = {
        'Cluster': i,
        'Size': len(cluster_data),
        'Avg_Age': cluster_data['Age'].mean(),
        'Pct_Male': cluster_data['Sex_encoded'].mean() * 100,
        'Top_Drug': drug_cols[np.argmax(cluster_data[drug_cols].mean())],
        'Pct_Migrated': cluster_data['migrated'].mean() * 100
    }
    # Add top 3 drugs
    drug_means = cluster_data[drug_cols].mean().sort_values(ascending=False)
    profile['Drug_1'] = f"{drug_means.index[0]} ({drug_means.values[0]*100:.1f}%)"
    profile['Drug_2'] = f"{drug_means.index[1]} ({drug_means.values[1]*100:.1f}%)"
    profile['Drug_3'] = f"{drug_means.index[2]} ({drug_means.values[2]*100:.1f}%)"
    cluster_profiles_C.append(profile)

profile_df_C = pd.DataFrame(cluster_profiles_C)
print("\nCluster Profiles (Option C):")
print(profile_df_C.to_string(index=False))

#run clusters for Demographics + Drugs + Migration, k=4
print('\n - 5. Drugs + Demographics + Migration clustering \n')

kmeans_D = KMeans(n_clusters=k, random_state=42, n_init=10)
clusters_D = kmeans_D.fit_predict(X_D_scaled)
df_cluster['Cluster_D'] = clusters_D

print(f"Cluster sizes (with migration feature, k={k}):")
print(df_cluster['Cluster_D'].value_counts().sort_index())

# Calculate cluster characteristics for Option D
cluster_profiles_D = []
for i in range(k):
    cluster_data = df_cluster[df_cluster['Cluster_D'] == i]
    profile = {
        'Cluster': i,
        'Size': len(cluster_data),
        'Avg_Age': cluster_data['Age'].mean(),
        'Pct_Male': cluster_data['Sex_encoded'].mean() * 100,
        'Pct_Migrated': cluster_data['migrated'].mean() * 100,
        'Top_Drug': drug_cols[np.argmax(cluster_data[drug_cols].mean())]
    }
    cluster_profiles_D.append(profile)

profile_df_D = pd.DataFrame(cluster_profiles_D)
print("\nCluster Profiles (Option D):")
print(profile_df_D.to_string(index=False))

#compare the 2.
print("\n- 6: Comparison -\n")

#check if migration significantly differs across clusters in Option C
print("Migration rate variation across Option C clusters:")
print(f"  Range: {profile_df_C['Pct_Migrated'].min():.1f}% - {profile_df_C['Pct_Migrated'].max():.1f}%")
print(f"  Std dev: {profile_df_C['Pct_Migrated'].std():.1f}")

print("\nMigration rate variation across Option D clusters (explicitly separated):")
print(f"  Range: {profile_df_D['Pct_Migrated'].min():.1f}% - {profile_df_D['Pct_Migrated'].max():.1f}%")
print(f"  Std dev: {profile_df_D['Pct_Migrated'].std():.1f}")

#silhouette scores for both options
sil_C = silhouette_score(X_C_scaled, clusters_C)
sil_D = silhouette_score(X_D_scaled, clusters_D)
print(f"\nSilhouette scores:")
print(f"  Option C: {sil_C:.3f}")
print(f"  Option D: {sil_D:.3f}")

#visualise clusters (PCA)
print("\n-  7: Visualisation -\n")

# PCA for 2D visualization
pca = PCA(n_components=2)
X_pca_C = pca.fit_transform(X_C_scaled)
X_pca_D = pca.fit_transform(X_D_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Option C
scatter1 = axes[0].scatter(X_pca_C[:, 0], X_pca_C[:, 1], c=clusters_C, cmap='viridis', alpha=0.5, s=10)
axes[0].set_title(f'Option C: Demographics + Drugs (k={k})')
axes[0].set_xlabel('PCA Component 1')
axes[0].set_ylabel('PCA Component 2')
plt.colorbar(scatter1, ax=axes[0], label='Cluster')

# Option D
scatter2 = axes[1].scatter(X_pca_D[:, 0], X_pca_D[:, 1], c=clusters_D, cmap='viridis', alpha=0.5, s=10)
axes[1].set_title(f'Option D: + Migration Flag (k={k})')
axes[1].set_xlabel('PCA Component 1')
axes[1].set_ylabel('PCA Component 2')
plt.colorbar(scatter2, ax=axes[1], label='Cluster')

plt.tight_layout()
plt.show()

#validation summary
print('\n - 8. Validation Summary -\n')

print("Statistical Validation:")
print(f"  Silhouette score (Option C): {sil_C:.3f} (0.3+ = decent separation)")
print(f"  Silhouette score (Option D): {sil_D:.3f}")
print(f"  All clusters have >50 members (stable)")

print("\nDomain logic check ")
print("  1. Are clusters separated by age? (young vs old)")
print("  2. Are there 'polydrug' clusters vs 'single drug' clusters?")
print("  3. Does migration create a distinct cluster?")
print("  4. Do the cluster profiles make real-world sense?")

print("\n Clustering complete. Review profiles above and choose a cluster.")

        

In [ ]:
#Refining option C.

print("=Refined: Option C (k=3) =\n")

k = 3
kmeans_C3 = KMeans(n_clusters=k, random_state=42, n_init=10)
clusters_C3 = kmeans_C3.fit_predict(X_C_scaled)
df_cluster['Cluster_C3'] = clusters_C3

#Cluster sizes
print(f"Cluster sizes (k={k}):")
print(df_cluster['Cluster_C3'].value_counts().sort_index())

#Silhouette score
sil_C3 = silhouette_score(X_C_scaled, clusters_C3)
print(f"\nSilhouette score (k=3): {sil_C3:.4f}")
print(f"Previous silhouette score (k=4): 0.1943")

#Calculate cluster profiles
cluster_profiles_C3 = []
for i in range(k):
    cluster_data = df_cluster[df_cluster['Cluster_C3'] == i]
    
    # Get top 3 drugs
    drug_means = cluster_data[drug_cols].mean().sort_values(ascending=False)
    
    profile = {
        'Cluster': i,
        'Size': len(cluster_data),
        'Pct_of_Total': f"{len(cluster_data)/len(df_cluster)*100:.1f}%",
        'Avg_Age': round(cluster_data['Age'].mean(), 1),
        'Pct_Male': f"{cluster_data['Sex_encoded'].mean() * 100:.1f}%",
        'Pct_Migrated': f"{cluster_data['migrated'].mean() * 100:.1f}%",
        'Top_Drug': drug_means.index[0],
        'Top_Drug_Pct': f"{drug_means.values[0]*100:.1f}%",
        'Drug_2': f"{drug_means.index[1]} ({drug_means.values[1]*100:.1f}%)",
        'Drug_3': f"{drug_means.index[2]} ({drug_means.values[2]*100:.1f}%)"
    }
    cluster_profiles_C3.append(profile)

profile_df_C3 = pd.DataFrame(cluster_profiles_C3)
print("\n=== CLUSTER PROFILES (k=3) ===\n")
print(profile_df_C3.to_string(index=False))

#compare k=3 v. k=4

print("\n=Compare: k=3 vs k=4 =\n")

# How did the clusters redistribute?
if 'Cluster_C' in df_cluster.columns:
    print("How k=4 clusters redistributed into k=3:")
    cross_tab = pd.crosstab(df_cluster['Cluster_C'], df_cluster['Cluster_C3'])
    print(cross_tab)

#visualise
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PCA visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_C_scaled)

# Plot 1: Clusters
scatter = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=clusters_C3, cmap='viridis', alpha=0.5, s=10)
axes[0].set_title(f'Option C Clusters (k=3)')
axes[0].set_xlabel('PCA Component 1')
axes[0].set_ylabel('PCA Component 2')
plt.colorbar(scatter, ax=axes[0], label='Cluster')

# Plot 2: Silhouette scores comparison
k_values = [2, 3, 4, 5, 6, 7, 8, 9, 10]
sil_values = [0.215, sil_C3, 0.194, 0.192, 0.190, 0.188, 0.185, 0.182, 0.181]
axes[1].plot(k_values, sil_values, 'ro-', linewidth=2, markersize=8)
axes[1].axvline(x=3, color='g', linestyle='--', alpha=0.7, label='k=3 (refined)')
axes[1].axvline(x=4, color='r', linestyle='--', alpha=0.5, label='k=4 (original)')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Scores: k=3 vs k=4')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n clustering complete.")


In [ ]:
#time series analysis. Has drug use evolved over time? 2012-2018

# Ensure year is available
if 'year' not in df.columns:
    df['year'] = pd.to_datetime(df['Date']).dt.year

# Filter to years 2012-2018
df_ts = df[df['year'].between(2012, 2018)].copy()

#1. deaths over time

deaths_by_year = df_ts.groupby('year').size()

plt.figure(figsize=(12, 6))
plt.plot(deaths_by_year.index, deaths_by_year.values, 'bo-', linewidth=2, markersize=8)
plt.fill_between(deaths_by_year.index, deaths_by_year.values, alpha=0.3)
plt.title('Total Drug Deaths in Connecticut (2012-2018)', fontsize=14, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Number of Deaths')
plt.grid(True, alpha=0.3)

# Add value labels
for x, y in zip(deaths_by_year.index, deaths_by_year.values):
    plt.text(x, y + 10, str(y), ha='center', fontweight='bold')

plt.tight_layout()

plt.show()

print("Total deaths by year")
print(deaths_by_year)
print(f"\nTotal increase: {deaths_by_year.iloc[-1] - deaths_by_year.iloc[0]} deaths from {deaths_by_year.index[0]} to {deaths_by_year.index[-1]}")

#2. drug prevalence

# Calculate prevalence per year for each drug
drug_trends = []
for drug in existing_drugs:
    yearly_pct = df_ts.groupby('year')[drug].mean() * 100
    for year, pct in yearly_pct.items():
        drug_trends.append({'Year': year, 'Drug': drug, 'Prevalence': pct})

drug_trends_df = pd.DataFrame(drug_trends)

# Plot top 6 drugs
top_6_drugs = df_ts[existing_drugs].mean().sort_values(ascending=False).head(6).index.tolist()

plt.figure(figsize=(14, 8))
for drug in top_6_drugs:
    drug_data = drug_trends_df[drug_trends_df['Drug'] == drug]
    plt.plot(drug_data['Year'], drug_data['Prevalence'], marker='o', linewidth=2, label=drug)

plt.title('Drug Prevalence Over Time (2012-2018)', fontsize=14, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Percentage of Deaths Involving Drug')
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#3. which drugs are on the up and up?

print("\n drug trends (2012 - 2018\n")

trend_summary = []
for drug in existing_drugs:
    pct_2012 = df_ts[df_ts['year'] == 2012][drug].mean() * 100
    pct_2018 = df_ts[df_ts['year'] == 2018][drug].mean() * 100
    change = pct_2018 - pct_2012
    trend_summary.append({
        'Drug': drug,
        '2012 (%)': round(pct_2012, 1),
        '2018 (%)': round(pct_2018, 1),
        'Change (pp)': round(change, 1),
        'Direction': '↑ Increased' if change > 0 else '↓ Decreased' if change < 0 else '→ Stable'
    })

trend_df = pd.DataFrame(trend_summary)
trend_df = trend_df.sort_values('Change (pp)', ascending=False)

print("BIGGEST INCREASES (2012 → 2018):")
print(trend_df.head(8).to_string(index=False))

print("\nBIGGEST DECREASES (2012 → 2018):")
print(trend_df.tail(8).to_string(index=False))

#4. fentanyl rise

plt.figure(figsize=(12, 6))

# Fentanyl trend
fent_data = drug_trends_df[drug_trends_df['Drug'] == 'Fentanyl']
plt.plot(fent_data['Year'], fent_data['Prevalence'], 'r-', linewidth=3, marker='s', markersize=8, label='Fentanyl')

# Heroin for comparison
heroin_data = drug_trends_df[drug_trends_df['Drug'] == 'Heroin']
plt.plot(heroin_data['Year'], heroin_data['Prevalence'], 'b-', linewidth=2, marker='o', markersize=6, label='Heroin')

plt.title('Fentanyl vs Heroin: The Opioid Transition (2012-2018)', fontsize=14, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Percentage of Deaths Involving Drug')
plt.legend()
plt.grid(True, alpha=0.3)

# Annotate crossover point
for year in [2014, 2015, 2016]:
    fent_val = fent_data[fent_data['Year'] == year]['Prevalence'].values
    heroin_val = heroin_data[heroin_data['Year'] == year]['Prevalence'].values
    if len(fent_val) > 0 and len(heroin_val) > 0 and fent_val[0] > heroin_val[0]:
        plt.annotate('Fentanyl surpasses Heroin', xy=(year, fent_val[0]), 
                     xytext=(year-0.5, fent_val[0]+10),
                     arrowprops=dict(arrowstyle='->', color='gray'))
        break

plt.tight_layout()
plt.show()

# Cluster Evolution (k=3)

# Get the index of rows used in clustering
cluster_rows = df_cluster.index  # These are the row indices that have cluster labels

# Create a mapping of row index to cluster
cluster_map = pd.Series(clusters_C3, index=cluster_rows)

# Add cluster column to df_ts (only for rows that exist in cluster_map)
df_ts['Cluster'] = df_ts.index.map(cluster_map)

# Check how many rows got clusters
print(f"Rows with cluster labels: {df_ts['Cluster'].notna().sum()}")
print(f"Rows without cluster labels (missing demographics): {df_ts['Cluster'].isna().sum()}")


# Cluster distribution over time (only rows with clusters)

df_ts_with_clusters = df_ts[df_ts['Cluster'].notna()].copy()

cluster_by_year = df_ts_with_clusters.groupby(['year', 'Cluster']).size().unstack(fill_value=0)
cluster_pct_by_year = cluster_by_year.div(cluster_by_year.sum(axis=1), axis=0) * 100

plt.figure(figsize=(12, 6))
for cluster in sorted(cluster_pct_by_year.columns):
    plt.plot(cluster_pct_by_year.index, cluster_pct_by_year[cluster], 
             marker='o', linewidth=2, label=f'Cluster {int(cluster)}')

plt.title('Victim Profile Distribution Over Time (2012-2018)', fontsize=14, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Percentage of Deaths')
plt.legend(title='Cluster')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n=== CLUSTER DISTRIBUTION OVER TIME (% of deaths per year) ===\n")
print(cluster_pct_by_year.round(1))


# Summary Statistics (without cluster evolution)


print("\n TIME SERIES SUMMARY \n")
print(f"Data period: 2012-2018")
print(f"Total deaths analyzed: {len(df_ts)}")
print(f"Year with most deaths: {deaths_by_year.idxmax()} ({deaths_by_year.max()} deaths)")
print(f"Year with fewest deaths: {deaths_by_year.idxmin()} ({deaths_by_year.min()} deaths)")

# Fastest growing drug 
if 'trend_df' in locals():
    fastest_grower = trend_df.iloc[0]
    print(f"\n🚀 Fastest growing drug: {fastest_grower['Drug']} (+{fastest_grower['Change (pp)']} percentage points)")

    fastest_decliner = trend_df.iloc[-1]
    print(f"📉 Fastest declining drug: {fastest_decliner['Drug']} ({fastest_decliner['Change (pp)']} percentage points)")

    # Fentanyl story
    fent_row = trend_df[trend_df['Drug'] == 'Fentanyl']
    if not fent_row.empty:
        fent_2012 = fent_row['2012 (%)'].values[0]
        fent_2018 = fent_row['2018 (%)'].values[0]
        print(f"\n💉 Fentanyl increased from {fent_2012}% to {fent_2018}% of deaths ({fent_2018/fent_2012:.1f}x increase)")